<a href="https://colab.research.google.com/github/Aksinhaa/ColabFold/blob/main/TGW_ANGSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**ANGSD**
 (Analysis of Next Generation Sequencing Data) is a software designed for analyzing low-coverage sequencing data without requiring explicit genotype calls. It uses probabilistic models to estimate genotype likelihoods directly from sequencing reads.

**Why SNP calling is done & its importance :**

SNP calling identifies single nucleotide variations between samples, which are the basis of genetic diversity analysis. These variants allow us to compare individuals and study evolutionary relationships and population structure.

In [ ]:
%%bash
# Install Miniconda
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

# Initialize conda
source /usr/local/miniconda/etc/profile.d/conda.sh

# Accept Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Add channels
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda create -y -n ngs_env angsd qualimap samtools plink

In [ ]:
%%bash
mkdir -p /content/workshop_data/aligned_bam
cd /content/workshop_data/aligned_bam
pwd

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/aligned_bam

wget https://zenodo.org/records/20049847/files/Hoggar1.bam
wget https://zenodo.org/records/20049847/files/Hoggar1.bam.bai
wget https://zenodo.org/records/20049847/files/Niger.bam
wget https://zenodo.org/records/20049847/files/Niger.bam.bai
wget https://zenodo.org/records/20049847/files/Senegal-P.bam
wget https://zenodo.org/records/20049847/files/Senegal-P.bam.bai

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/aligned_bam

for sample in Hoggar1 Niger Senegal-P
do
  echo "Filtering $sample..."

  samtools view \
  -h \
  -F 4 \
  -q 30 \
  -b \
  ${sample}.bam \
  > ${sample}.filtered.bam

done

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Indexing filtered $sample..."

  samtools index aligned_bam/${sample}.filtered.bam

done

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Removing duplicates for $sample..."

  samtools markdup \
  -r \
  -s \
  aligned_bam/${sample}.filtered.bam \
  aligned_bam/${sample}.filtered.dedup.bam

done

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Indexing deduplicated $sample..."

  samtools index aligned_bam/${sample}.filtered.dedup.bam

done

This step evaluates the quality of the final aligned BAM files using Qualimap (`bamqc`).

The input to Qualimap is the filtered and deduplicated BAM file.

The `-bam` option specifies the input file, `-nt 2` enables multithreading for faster execution, and `-outdir` defines a separate output folder for each sample to keep results organized. The `-outformat HTML` generates an interactive report that can be easily viewed in a browser.

Qualimap provides detailed metrics such as coverage distribution, mapping quality, GC bias, and read distribution across the genome. This step is important because it gives a comprehensive overview of alignment quality after all preprocessing steps, helping to identify any remaining biases or issues before proceeding to variant calling and downstream analyses.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

mkdir -p /content/workshop_data/qualimap_results

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Running Qualimap BAMQC for $sample..."

  conda run -n ngs_env qualimap bamqc \
    -bam aligned_bam/${sample}.filtered.dedup.bam \
    -nt 2 \
    -outdir qualimap_results/${sample} \
    -outformat HTML

done

This step prepares the input required for downstream analysis with ANGSD by creating a list of BAM files. The command ls `aligned_bam/*.filtered.dedup.bam > bam.list `collects the paths of all processed BAM files and saves them into a single text file. Tools like ANGSD do not take multiple BAM files directly; instead, they require a file containing a list of input files.

The `cat bam.list` command is used to verify that all expected samples are correctly included in the list. This step is important because it ensures that all samples are analyzed together, enabling comparative analyses such as SNP calling and population genetics. Without this file, ANGSD would not be able to process multiple samples in a single run.

In [ ]:
%%bash
cd /content/workshop_data

ls aligned_bam/*.filtered.dedup.bam > bam.list

cat bam.list

We need to download the reference file and its index for the next step.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env
mkdir -p /content/workshop_data/reference

cd /content/workshop_data/reference

wget https://zenodo.org/records/19947652/files/chr31.fasta
wget https://zenodo.org/records/19947652/files/chr31.fasta.amb
wget https://zenodo.org/records/19947652/files/chr31.fasta.ann
wget https://zenodo.org/records/19947652/files/chr31.fasta.bwt
wget https://zenodo.org/records/19947652/files/chr31.fasta.dict
wget https://zenodo.org/records/19947652/files/chr31.fasta.fai
wget https://zenodo.org/records/19947652/files/chr31.fasta.pac
wget https://zenodo.org/records/19947652/files/chr31.fasta.sa

This step performs SNP calling using ANGSD, which is specifically designed for low-coverage sequencing data such as ancient DNA. The tool takes a list of BAM files (`bam.list`) and the reference genome to infer genetic variants across all samples simultaneously. The output is stored in the `angsd_results` directory with the prefix `Pigeons_snp`, generating files such as `.haplo.gz`, `.log`, and `.arg`.

The parameter `-doHaploCall 1` enables pseudo-haploid SNP calling, where one allele is randomly sampled per site—this is crucial for aDNA because coverage is often too low for reliable diploid genotype calling. The `-GL 2` option uses a GATK-like model to estimate genotype likelihoods, accounting for sequencing errors. Additional filters like `-minMapQ 30` and `-minQ 20` ensure that only high-quality reads and bases are considered, while `-uniqueOnly 1` and `-remove_bads 1` remove ambiguous and low-quality alignments.

The `-doCounts 1` option calculates read depth and allele counts, and `-doMajorMinor 1` infers the major and minor alleles at each site. The `-skipTriallelic 1` ensures only biallelic SNPs are retained, which simplifies downstream analysis.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

mkdir -p angsd_results

angsd \
  -bam bam.list \
  -out angsd_results/Pigeons_snp \
  -ref reference/chr31.fasta \
  -doHaploCall 1 \
  -doCounts 1 \
  -nThreads 2 \
  -minMapQ 30 \
  -minQ 20 \
  -uniqueOnly 1 \
  -remove_bads 1 \
  -skipTriallelic 1 \
  -doMajorMinor 1 \
  -GL 2

This step converts the SNP output from ANGSD into a format compatible with PLINK using the haploToPlink utility. The input file Pigeons_snp.haplo.gz contains pseudo-haploid SNP data generated from low-coverage sequencing. The command transforms this into PLINK text files (.tped and .tfam), which are required for downstream population genetic analyses. This conversion is important because tools like PLINK cannot directly read ANGSD output formats

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/angsd_results

haploToPlink Pigeons_snp.haplo.gz Pigeons_snp


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda create -y -n ngs_env plink

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/angsd_results

plink \
  --tfile Pigeons_snp \
  --make-bed \
  --out Pigeons_snp \
  --allow-extra-chr

This step performs SNP filtering using PLINK to improve the quality of the dataset before downstream analysis. The `--bfile Pigeons_snp` option loads the binary PLINK files generated earlier, and `--maf 0.01` removes variants with a minor allele frequency less than 1%, which are often uninformative or prone to error. The `--make-bed` command creates a new set of filtered binary files, saved with the prefix `Pigeons_snp_mf01`. The `--allow-extra-chr` option ensures compatibility with non-standard chromosome names.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/angsd_results

plink \
  --bfile Pigeons_snp \
  --maf 0.01 \
  --make-bed \
  --out Pigeons_snp_mf01 \
  --allow-extra-chr